# 11. Pressure over the possession (reception → pass) vs the analysts' labels

Question: the analysts' pass-level pressure label seems to reflect the possession, not only the pass frame.
Is the label better explained by **how much pressure the carrier was exposed to between receiving and passing**
than by the distance at the pass frame alone? And are labels that persist from the reception backed by real
exposure, or do they persist without any defender coming close?

Per open-play pass (all 64 matches), from the tracking data (raw coordinates, 29.97 fps), window = reception
(game event start frame) → pass frame, capped at 15 s. The carrier is identified in the tracking file as the track
closest to the event-file passer position at the pass frame; defenders are the tracks of the other team.
Measures of the nearest-defender distance over the window:
- `d_start` (reception), `d_end` (pass), `d_min` (minimum), `d_mean` (mean)
- `t_under_274`, `share_under_274`: time / share of frames with a defender within 2.74 m (PFF 'pressured' guide)
- `t_under_457`, `share_under_457`: same for 4.57 m (PFF 'attempted pressure' guide)
- `onset_274`: time from reception to the first frame within 2.74 m

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from concurrent.futures import ProcessPoolExecutor
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, SplineTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

from football.common import KEY, cv_predict, boot_ci, boot_diff
from football.tracking import possession_exposure

OUT = os.path.join("..", "outputs")
FIG = os.path.join(OUT, "figs")
E = pd.read_csv(os.path.join(OUT, "pressure_events.csv.gz"))
R = pd.read_csv(os.path.join(OUT, "reception_labels.csv.gz"))
print(f"open-play passes: {len(E):,}")

open-play passes: 61,012


## 1. Exposure over the possession (cached, ~8 min)

In [2]:
CACHE = os.path.join(OUT, "possession_exposure.csv.gz")
if not os.path.exists(CACHE):
    jobs = {g: set(d.pe_id.astype(int)) for g, d in E.groupby("game_id")}
    with ProcessPoolExecutor(6) as ex:
        parts = list(ex.map(possession_exposure, jobs, jobs.values()))
    pd.DataFrame([r for p in parts for r in p]).to_csv(CACHE, index=False)
X = E.merge(pd.read_csv(CACHE), on=KEY).merge(R, on=KEY, how="left")
X["dur"] = X.n_frames / 29.97
X["fail"] = (~X.complete.astype(bool)).astype(int)
print(f"passes with exposure: {len(X):,} of {len(E):,};  possession duration median {X.dur.median():.2f} s;  "
      f"one-touch (1 frame): {(X.n_frames <= 1).mean():.1%}")
X.groupby("label")[["d_start", "d_min", "d_mean", "d_end", "t_under_274", "share_under_274", "dur"]].median().round(2)

passes with exposure: 60,483 of 61,012;  possession duration median 1.17 s;  one-touch (1 frame): 28.3%


,d_start,d_min,d_mean,d_end,t_under_274,share_under_274,dur
label,,,,,,,
A,5.58,3.49,4.61,3.56,0.00,0.00,0.97
L,6.65,2.98,4.84,3.12,0.00,0.00,1.60
N,8.90,6.23,7.87,6.58,0.00,0.00,1.30
P,3.74,2.03,3.02,2.20,0.03,0.39,0.90


## 2. Which measure best reproduces the pass label? (P vs N, game-grouped CV, cluster bootstrap by game)

In [3]:
PN = X[X.label.isin(["P", "N"])].copy()
y, g = (PN.label == "P").astype(int), PN.game_id
single = {"distance at the pass (d_end)": -PN.d_end, "distance at reception (d_start)": -PN.d_start,
          "minimum distance over possession (d_min)": -PN.d_min, "mean distance over possession (d_mean)": -PN.d_mean,
          "time within 2.74 m": PN.t_under_274, "share of possession within 2.74 m": PN.share_under_274,
          "share of possession within 4.57 m": PN.share_under_457}
rows = []
for k, s in single.items():
    lo, hi = boot_ci(y, s, roc_auc_score, n=300, groups=g)
    rows.append(dict(measure=k, roc_auc=roc_auc_score(y, s), ci95=f"[{lo:.3f}, {hi:.3f}]"))
single_tab = pd.DataFrame(rows).round(3)
single_tab

,measure,roc_auc,ci95
0,distance at the pass (d_end),0.866,"[0.846, 0.882]"
1,distance at reception (d_start),0.788,"[0.772, 0.800]"
2,minimum distance over possession (d_min),0.853,"[0.834, 0.867]"
3,mean distance over possession (d_mean),0.835,"[0.817, 0.849]"
4,time within 2.74 m,0.777,"[0.762, 0.790]"
5,share of possession within 2.74 m,0.788,"[0.772, 0.801]"
6,share of possession within 4.57 m,0.805,"[0.788, 0.819]"


In [4]:
CONT = {"d_end", "d_start", "d_min", "d_mean", "t_under_274", "share_under_274", "share_under_457", "dur"}


def mk(cols):
    ct = ColumnTransformer([(c, SplineTransformer(n_knots=4), [c]) if c in CONT else (c, "passthrough", [c]) for c in cols])
    return lambda s: make_pipeline(ct, StandardScaler(), LogisticRegression(max_iter=3000))


MODELS = {"I  pass frame only": ["d_end"],
          "II + possession exposure": ["d_end", "d_min", "d_mean", "share_under_274", "share_under_457", "dur"],
          "III + reception label (human)": ["d_end", "d_min", "d_mean", "share_under_274", "share_under_457", "dur", "rec_P", "rec_A", "rec_L"]}
for lab in "PAL":
    PN[f"rec_{lab}"] = (PN.init_pressure == lab).astype(int)
preds, rows, prev = {}, [], None
for name, cols in MODELS.items():
    preds[name] = cv_predict(mk(cols), PN[cols].astype(float), y, g, seeds=(0,))
    row = dict(model=name, roc_auc=roc_auc_score(y, preds[name]))
    if prev:
        d, ci, _ = boot_diff(y, preds[name], preds[prev], roc_auc_score, n=300, groups=g)
        row.update(gain=d, ci95=f"[{ci[0]:+.4f}, {ci[1]:+.4f}]")
    rows.append(row); prev = name
nested = pd.DataFrame(rows).round(4)
nested

,model,roc_auc,gain,ci95
0,I pass frame only,0.8632,NaN,NaN
1,II + possession exposure,0.8779,0.0147,"[+0.0123, +0.0173]"
2,III + reception label (human),0.9139,0.0362,"[+0.0310, +0.0424]"


## 3. Labels that persist from the reception: backed by real exposure?
Passes labelled P whose nearest defender is ≥ 4 m away at the pass ('far P'), split by the reception label and by whether
a defender was within 2.74 m at any time during the possession.

In [5]:
PN["far"] = PN.d_end >= 4
PN["exposed"] = PN.d_min < 2.74
far = PN[PN.far]
t = far.pivot_table(index=["init_pressure"], columns="label", values="exposed", aggfunc="mean").round(3)
n = far.pivot_table(index=["init_pressure"], columns="label", values="exposed", aggfunc="size")
display(pd.concat([t.add_prefix("share exposed | pass="), n.add_prefix("n | pass=")], axis=1).loc[["N", "P"]])
fp = far[far.label == "P"]
groups = pd.Series(np.select([fp.exposed, fp.init_pressure.eq("P")],
                             ["defender within 2.74 m earlier in the possession", "reception labelled P, no defender within 2.74 m"],
                             "neither"), index=fp.index)
far_breakdown = groups.value_counts().rename("n").to_frame()
far_breakdown["share of far P"] = (far_breakdown.n / len(fp)).round(3)
far_breakdown["share of all P"] = (far_breakdown.n / y.sum()).round(3)
far_breakdown

label,share exposed | pass=N,share exposed | pass=P,n | pass=N,n | pass=P
init_pressure,,,,
N,0.023,0.083,23604,2272
P,0.506,0.091,399,1896


,n,share of far P,share of all P
neither,2104,0.502,0.095
"reception labelled P, no defender within 2.74 m",1724,0.411,0.078
defender within 2.74 m earlier in the possession,365,0.087,0.016


In [6]:
# P-rate at the pass by exposure during the possession, within bins of the pass-frame distance
PN["dbin"] = pd.cut(PN.d_end, [0, 2, 3, 4, 6, 100])
PN["expo"] = pd.cut(PN.share_under_274, [-0.01, 0, 0.25, 0.5, 1.0], labels=["none", "≤25%", "25–50%", ">50%"])
expo_tab = PN.pivot_table(index="dbin", columns="expo", values="label", aggfunc=lambda s: (s == "P").mean(), observed=True).round(3)
expo_tab

expo,none,≤25%,25–50%,>50%
dbin,,,,
"(0, 2]",NaN,0.927,0.947,0.892
"(2, 3]",0.644,0.831,0.767,0.752
"(3, 4]",0.446,0.351,0.432,0.468
"(4, 6]",0.201,0.380,0.333,0.361
"(6, 100]",0.114,0.232,0.320,0.467


## 4. Pass failure: instant vs possession exposure vs human label

In [7]:
O = PN.dropna(subset=["pass_dist"]).copy()
BASE = ["pass_dist", "dx", "lane_min", "target_near", "cx", "cy"]
CONT.update(BASE)
O["h_P"] = (O.label == "P").astype(int)
fm = {"base: pass geometry": BASE,
      "+ distance at the pass": BASE + ["d_end"],
      "+ possession exposure": BASE + ["d_end", "d_min", "d_mean", "share_under_274", "share_under_457", "dur"],
      "+ human label": BASE + ["d_end", "d_min", "d_mean", "share_under_274", "share_under_457", "dur", "h_P"]}
fp_, rows = {}, []
for k, cols in fm.items():
    fp_[k] = cv_predict(mk(cols), O[cols].astype(float), O.fail, O.game_id, seeds=(0,))
names = list(fm)
for i, k in enumerate(names):
    row = dict(model=k, roc_auc=roc_auc_score(O.fail, fp_[k]))
    if i:
        d, ci, _ = boot_diff(O.fail, fp_[k], fp_[names[i - 1]], roc_auc_score, n=300, groups=O.game_id)
        row.update(gain=d, ci95=f"[{ci[0]:+.4f}, {ci[1]:+.4f}]")
    rows.append(row)
fail_tab = pd.DataFrame(rows).round(4)
print(f"fail rate {O.fail.mean():.3f}, n = {len(O):,}")
fail_tab

fail rate 0.166, n = 52,327


,model,roc_auc,gain,ci95
0,base: pass geometry,0.8417,NaN,NaN
1,+ distance at the pass,0.8443,0.0027,"[+0.0018, +0.0035]"
2,+ possession exposure,0.8566,0.0122,"[+0.0109, +0.0136]"
3,+ human label,0.8576,0.0010,"[+0.0005, +0.0014]"


## 5. Figure

In [8]:
fig, ax = plt.subplots(figsize=(5.4, 3.5))
x = [iv.mid if iv.right < 100 else 8 for iv in expo_tab.index]
for col, c in zip(expo_tab.columns, ["#c6dbef", "#6baed6", "#2171b5", "#08306b"]):
    ax.plot(x, expo_tab[col], marker="o", color=c, label=f"within 2.74 m for {col} of the possession")
ax.set_xlabel("nearest defender distance at the pass (m)"); ax.set_ylabel("share labelled 'pressured'")
ax.set_ylim(0, 1); ax.legend(frameon=False, fontsize=7)
ax.set_title("Same distance at the pass, different pressure history", fontsize=9)
fig.tight_layout(); fig.savefig(os.path.join(FIG, "fig_possession_exposure.png"), dpi=300)

## Save

In [9]:
single_tab.to_csv(os.path.join(OUT, "exposure_single_auc.csv"), index=False)
nested.to_csv(os.path.join(OUT, "exposure_nested.csv"), index=False)
far_breakdown.to_csv(os.path.join(OUT, "exposure_far_breakdown.csv"))
expo_tab.to_csv(os.path.join(OUT, "exposure_prate_table.csv"))
fail_tab.to_csv(os.path.join(OUT, "exposure_failure.csv"), index=False)
print("saved")

saved
